### Step 1: Import Required Libraries
Here, we import all necessary Python libraries for data handling, model building, training, and visualization.

In [ ]:

import torch
import torch.nn as nn
import torch.optim as optim
from tqdm.auto import tqdm
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
import matplotlib.pyplot as plt


### Step 2: Load and Explore the Dataset
We use Scikit-learn’s built-in **California Housing dataset**, which contains information about various housing attributes and their prices.

In [ ]:

device = 'cpu'
data = fetch_california_housing()
X, y = data.data, data.target
print(f"Dataset Shape: {X.shape}, Target Shape: {y.shape}")
print(f"Feature Names: {data.feature_names}")


### Step 3: Split the Dataset
We split the dataset into **training** and **testing** sets (80% training, 20% testing) to evaluate model performance properly.

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


### Step 4: Standardize the Data
We scale the features using **StandardScaler** to normalize them for better neural network training stability.

In [ ]:

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


### Step 5: Convert Data to PyTorch Tensors
The data is converted into **torch tensors** to feed it into the neural network.

In [ ]:

X_train_t = torch.FloatTensor(X_train).to(device)
X_test_t = torch.FloatTensor(X_test).to(device)
y_train_t = torch.FloatTensor(y_train).view(-1, 1).to(device)
y_test_t = torch.FloatTensor(y_test).view(-1, 1).to(device)


### Step 6: Create DataLoader
We use **DataLoader** to feed data in mini-batches for efficient training.

In [ ]:

train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, shuffle=True, batch_size=256)


### Step 7: Define the Neural Network Architecture
We create a feed-forward neural network with multiple hidden layers and dropout regularization.

In [ ]:

class HousingNET(nn.Module):
    def __init__(self, input_dim):
        super(HousingNET, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.net(x)


### Step 8: Initialize Model, Loss Function, and Optimizer
We define our model, use **Mean Squared Error (MSE)** as the loss, and **Adam optimizer** for training.

In [ ]:

model = HousingNET(X_train.shape[1]).to(device)
loss_fn = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


### Step 9: Train the Model
We train the model for multiple epochs, logging training and testing losses, **RMSE**, and **R²** each epoch.

In [ ]:

EPOCHS = 50
train_losses, test_losses = [], []

print("Training California Housing Price Prediction Model...")
print(f"{'Epoch':<10}{'Train Loss':<15}{'Test Loss':<15}{'RMSE':<15}{'R²':<10}")
print("-" * 65)

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, total_samples = 0.0, 0

    for X_batch, y_batch in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=False):
        optimizer.zero_grad()
        preds = model(X_batch)
        loss = loss_fn(preds, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * X_batch.size(0)
        total_samples += X_batch.size(0)

    avg_train_loss = total_loss / total_samples
    train_losses.append(avg_train_loss)

    # Evaluate
    model.eval()
    with torch.no_grad():
        test_preds = model(X_test_t)
        test_loss = loss_fn(test_preds, y_test_t).item()
        test_losses.append(test_loss)
        rmse = np.sqrt(test_loss)
        ss_res = torch.sum((y_test_t - test_preds) ** 2).item()
        ss_tot = torch.sum((y_test_t - torch.mean(y_test_t)) ** 2).item()
        r2_score = 1 - (ss_res / ss_tot)

    print(f"{epoch:<10}{avg_train_loss:<15.4f}{test_loss:<15.4f}{rmse:<15.4f}{r2_score:<10.4f}")


### Step 10: Evaluate Final Model Performance
We evaluate the trained model on the test set and calculate RMSE, R² score, and MAE.

In [ ]:

model.eval()
with torch.no_grad():
    final_preds = model(X_test_t)
    final_rmse = np.sqrt(loss_fn(final_preds, y_test_t)).item()
    ss_res = torch.sum((y_test_t - final_preds) ** 2).item()
    ss_tot = torch.sum((y_test_t - torch.mean(y_test_t)) ** 2).item()
    final_r2 = 1 - (ss_res / ss_tot)
    mae = torch.mean(torch.abs(y_test_t - final_preds)).item()

print("\n" + "="*65)
print("FINAL METRICS:")
print(f"  RMSE: {final_rmse:.4f}")
print(f"  R² Score: {final_r2:.4f}")
print(f"  Mean Absolute Error: {mae:.4f}")
print("="*65)


### Step 11: Visualize Training Progress and Predictions
We visualize how the model’s training and testing loss evolved and how well predictions align with actual prices.

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss plot
axes[0].plot(train_losses, label='Train Loss', alpha=0.8)
axes[0].plot(test_losses, label='Test Loss', alpha=0.8)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Training and Test Loss Over Time')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Predictions vs Actual
axes[1].scatter(y_test_t.numpy(), final_preds.numpy(), alpha=0.5)
axes[1].plot([y_test_t.min(), y_test_t.max()], [y_test_t.min(), y_test_t.max()], 'r--', lw=2, label='Perfect Prediction')
axes[1].set_xlabel('Actual Price')
axes[1].set_ylabel('Predicted Price')
axes[1].set_title(f'Predictions vs Actual (R²={final_r2:.3f})')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('housing_price_results.png', dpi=150, bbox_inches='tight')
print("\nVisualization saved as 'housing_price_results.png'")
plt.show()


### Step 12: Save the Trained Model
Finally, we save the trained model for future inference.

In [ ]:

torch.save(model.state_dict(), 'housing_price_model.pth')
print("Model saved as 'housing_price_model.pth'")
